In [0]:
# =============================================================================
# NOTEBOOK  : silver_pos_transactions
# PURPOSE   : bronze.pos_transactions → silver.pos_transactions
# LAYER     : Silver (clean, type-cast, dedup, merge)
# FREQUENCY : Continuous micro-batch (JOB A — runs 24/7)
# TRIGGER   : availableNow (dev) / no trigger (prod)
# NOTE      : Handles BOTH stream and batch rows from bronze in one notebook
#             _source column distinguishes row origin:
#               "pos_realtime_stream" → Kafka stream rows
#               "pos_batch_csv"       → daily CSV batch rows
#             MERGE POSSIBLE CASES: Case 1 means only stream data (regular hours), Case 2 means stream + batch data (midnight hours)
#               Case 1.1 → transaction_id NOT in silver, source = stream    → INSERT
#               Case 1.2 → transaction_id IN silver, source = stream        → IGNORE (no update)
#               Case 2.1 → transaction_id NOT in silver, source = stream    → INSERT (same as 1.1)
#               Case 2.2 → transaction_id IN silver, source = stream        → IGNORE (same as 1.2)
#               Case 2.3 → transaction_id NOT in silver, source = batch     → INSERT
#               Case 2.4 → transaction_id IN silver, source = batch, data different → UPDATE
#               Case 2.5 → transaction_id IN silver, source = batch, data same     → IGNORE
#             MERGE rules:
#               Case 1.1 / 2.1 / 2.3 → new transaction_id → INSERT
#               Case 1.2 / 2.2        → duplicate stream row → IGNORE
#               Case 2.4              → batch row with data change → UPDATE
#               Case 2.5              → batch row, no change → IGNORE
#             Two writers never write to silver simultaneously:
#               JOB A  → only writer to silver (continuous)
#               JOB B  → only writes to bronze, then restarts JOB A
# =============================================================================

# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")

from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, to_timestamp, when
)
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable

with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)

SOURCE_TABLE = cfg["tables"]["bronze_pos_transactions"]
TARGET_TABLE = cfg["tables"]["silver_pos_transactions"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_pos_transactions"]
PIPELINE     = "silver_pos_transactions"

In [0]:
# ── foreachBatch function + Stream ────────────────────────────────────

def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch.
    micro_batch_df : regular batch DataFrame — all new bronze rows in this window
    microbatch_id  : unique identifier for this micro-batch (for logging)

    Both stream and batch rows flow through here.
    _source column tells us which is which.
    MERGE rules handle each case correctly without any explicit if-else.
    """

    # ── SKIP EMPTY MICRO-BATCHES ──────────────────────────────────────────────
    # During quiet periods or right after JOB A restarts, bronze may have
    # no new rows. Skip silently — no audit entry for empty batches.
    if micro_batch_df.isEmpty():
        print(f"[SKIP] microbatch_id={microbatch_id} — no new rows")
        return

    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)

    try:
        rows_read = micro_batch_df.count()

        # ── TRANSFORM ─────────────────────────────────────────────────────────
        # Rules:
        #   - Cast timestamp ISO string → TimestampType, rename to transaction_ts
        #   - Cast money columns Double → Decimal(10,2)
        #   - Derive transaction_date → partition column (event time, not ingest time)
        #   - Add silver audit columns
        #   - Select only silver schema columns — drops all bronze-only columns

        df = (
            micro_batch_df

            # 1. Cast timestamp ISO string → TimestampType
            #    Source format: "2023-08-19T22:26:11Z"
            #    Renamed to transaction_ts — avoids Spark reserved word 'timestamp'
            .withColumn(
                "transaction_ts", to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
            )

            # 2. Money columns → Decimal for precision
            #    Double has floating point imprecision — critical for financial data
            .withColumn("unit_price",   col("unit_price").cast(DecimalType(10, 2)))
            .withColumn("total_amount", col("total_amount").cast(DecimalType(10, 2)))

            # 3. transaction_date derived from event time (transaction_ts)
            #    NOT from ingested_at — silver partitions by when event happened
            # .withColumn("transaction_date", to_date(to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")).cast("string"))
            .withColumn("transaction_date", to_date(col("transaction_ts")).cast("string"))

            # 4. Silver audit columns
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   col("_source"))  # carry _source into silver

            # 5. Enforce silver schema — drops bronze-only cols in one line
            #    (_source, source_file, ingested_date, etc.)
            .select([f.name for f in SILVER_POS_TRANSACTIONS_SCHEMA.fields])
        )

        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Single MERGE handles all 5 cases:
        #
        # whenMatchedUpdate (Case 2.4):
        #   → matched on transaction_id
        #   → source is batch (CSV daily export)
        #   → at least one business field is different from silver
        #   → UPDATE only changed business fields
        #   → transaction_ts NOT updated — stream event time is authoritative
        #
        # whenNotMatchedInsertAll (Cases 1.1, 2.1, 2.3):
        #   → transaction_id not in silver yet
        #   → INSERT regardless of source (stream or batch)
        #
        # No rule fires (Cases 1.2, 2.2, 2.5):
        #   → matched on transaction_id
        #   → source is stream (duplicate) → no whenMatchedUpdate for stream
        #   → source is batch but no data change → condition fails
        #   → Delta does nothing → row effectively IGNORED

        silver_table = DeltaTable.forName(spark, TARGET_TABLE)

        (
            silver_table.alias("t")
            .merge(
                df.alias("s"),
                "t.transaction_id = s.transaction_id"
            )

            # Case 2.4 — batch row with actual data correction
            # Only update fields that the batch can authoritatively correct
            # transaction_ts is intentionally excluded — stream time is real event time
            .whenMatchedUpdate(
                condition="""
                    s.source_system = 'pos_batch_csv'
                    AND (
                        t.total_amount    != s.total_amount    OR
                        t.quantity        != s.quantity        OR
                        t.payment_method  != s.payment_method  OR
                        t.channel         != s.channel
                    )
                """,
                set={
                    "total_amount":    "s.total_amount",
                    "quantity":        "s.quantity",
                    "payment_method":  "s.payment_method",
                    "channel":         "s.channel",
                    "processed_at":    "current_timestamp()",
                    "pipeline_run_id": f"'{run_id}'"
                }
            )

            # Cases 1.1, 2.1, 2.3 — new transaction, insert regardless of source
            .whenNotMatchedInsertAll()

            .execute()
        )

        rows_written = df.count()
        print(f"[DONE] microbatch_id={microbatch_id} | "
              f"rows_read={rows_read} | rows_written={rows_written}")

        # ── END AUDIT (SUCCESS) ───────────────────────────────────────────────
        end_audit(
            spark, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_written
        )

    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise


# ── READ FROM BRONZE AS STREAM ────────────────────────────────────────────────
# Delta streaming reads only NEW rows added to bronze since last checkpoint
# When JOB A restarts after batch window:
#   → Kafka backlog rows land in bronze via stream notebook
#   → Batch CSV rows already in bronze from JOB B
#   → Both flow through here naturally — no special handling needed
# checkpoint ensures no row is processed twice across restarts
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")  # safe guard — bronze is append-only
    .table(SOURCE_TABLE)
)

# ── WRITE WITH foreachBatch ───────────────────────────────────────────────────
# trigger(availableNow=True) for dev — process all pending rows and stop
# Remove trigger entirely for prod JOB A — runs as continuous micro-batch 24/7
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)   # ← remove this line for prod continuous mode
    .start()
)

query.awaitTermination()

In [0]:
%sql
-- I want to see audit table complete
-- SELECT * FROM azure3_team7_project.platform.pipeline_audit;
-- SELECT COUNT(*) FROM azure3_team7_project.silver.pos_transactions;
SELECT * FROM azure3_team7_project.bronze.pos_transactions WHERE transaction_id = "TXN_00000001";

In [0]:
%sql
-- SELECT * FROM azure3_team7_project.platform.pipeline_audit;

In [0]:
# Run this in a separate cell — bypass foreachBatch entirely
# Read bronze as batch, apply same transforms, check result

micro_batch_df = (
    spark.read.table("azure3_team7_project.bronze.pos_transactions")
    .limit(10)
)

# Apply same transforms as inside process_microbatch
df = (
    micro_batch_df
    .withColumn("transaction_ts",   to_timestamp(col("timestamp")))
    .withColumn("unit_price",       col("unit_price").cast(DecimalType(10,2)))
    .withColumn("total_amount",     col("total_amount").cast(DecimalType(10,2)))
    .withColumn("transaction_date", to_date(col("transaction_ts")).cast("string"))
    .withColumn("processed_at",     current_timestamp())
    .withColumn("pipeline_run_id",  lit("debug"))
    .withColumn("source_system",    col("_source"))
    .select([f.name for f in SILVER_POS_TRANSACTIONS_SCHEMA.fields])
)

df.printSchema()
df.select("transaction_id", "source_system", "transaction_date").show(5, truncate=False)

# Check silver
spark.read.table("azure3_team7_project.silver.pos_transactions") \
    .select("transaction_id").show(5, truncate=False)

In [0]:
# Check if ANY transaction_id overlaps between stream and batch in bronze
stream_df = (
    spark.read.table("azure3_team7_project.bronze.pos_transactions")
    .filter(col("_source") == "pos_realtime_stream")
    .select("transaction_id")
)

batch_df = (
    spark.read.table("azure3_team7_project.bronze.pos_transactions")
    .filter(col("_source") == "pos_batch_csv")
    .select("transaction_id")
)

overlap = stream_df.join(batch_df, "transaction_id", "inner").count()
print(f"Overlapping transaction_ids: {overlap}")

In [0]:
# reverse the silver pipeline run

spark.sql("TRUNCATE TABLE azure3_team7_project.silver.pos_transactions");
spark.sql("TRUNCATE TABLE azure3_team7_project.bronze.pos_transactions");

dbutils.fs.rm(
    "/Volumes/azure3_team7_project/bronze/checkpoints/pos_transactions_stream",
    recurse=True
);

dbutils.fs.rm(
    "/Volumes/azure3_team7_project/bronze/checkpoints/pos_transactions_batch",
    recurse=True
);

dbutils.fs.rm(
    "/Volumes/azure3_team7_project/silver/checkpoints/pos_transactions_stream/",
    recurse=True
);

dbutils.fs.rm(
    "/Volumes/azure3_team7_project/silver/checkpoints/pos_transactions_batch/",
    recurse=True
);

# clear cache
spark.sql("CLEAR CACHE");

In [0]:
%sql
SELECT COUNT(*) FROM azure3_team7_project.silver.pos_transactions;